# DPM Knowledge Graph — BCC Live Demo

This notebook demonstrates the five core capabilities of the DPM assistant:
**semantic search, metadata filtering, domain Q&A, workflow guidance, and literature lookup.**


---
### How this system works (big picture)

The DPM assistant sits on top of a **Neo4j knowledge graph**, a database that stores
porous media datasets as nodes (like `DigitalDataset`, `Sample`, `RelatedPublication`)
and connects them with relationships (like `HAS_PUBLICATION`, `HAS_CORE`).

When a user asks a question, the assistant figures out which *type* of question it is
(called **intent classification**) and routes it to the right tool:
- Questions about *similar datasets* → vector search in Neo4j
- Questions about *specific properties* → structured Cypher query
- Conceptual questions → answered directly by the LLM
- How-to questions → workflow guidance chain
- Paper/citation questions → literature lookup in the graph

`Neo4jGraphStore` is the Python class that handles all the database communication.
It lives in `graph_store.py` and is what we import below.

In [8]:
import sys
print(sys.executable)

/Users/juliashannon/dpm_rocco_curator/.venv/bin/python


In [14]:
# --- IMPORTS AND CONNECTION SETUP ---
#
# os: lets us read environment variables (like the DB password) from the .env file
# dotenv: loads the .env file into the environment so os.getenv() can find the values
# Neo4jGraphStore: our custom class that wraps the Neo4j driver

import os
from dotenv import load_dotenv
from graph_store import Neo4jGraphStore

# Reads NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, USE_NEO4J from the .env file.
# Must be called before Neo4jGraphStore() so the values are available.
load_dotenv()

# Opens a connection pool to Neo4j and verifies the handshake immediately.
# If this line errors, the DB is unreachable, check VPN or .env values.
store = Neo4jGraphStore()

# Quick sanity check: print the URI so we know which DB we're talking to,
# and fetch the schema to confirm nodes and relationships exist.
print("Connected to Neo4j at:", os.getenv("NEO4J_URI"))
print("Schema:", store.get_schema_blueprint())

Connected to Neo4j at: None
Schema: {'node_labels': ['DigitalDataset'], 'relationship_types': []}


---
## Query 1 — Semantic Search

**Intent:** Find datasets by *conceptual similarity*, not exact keyword match.

> *"Find datasets similar to micro-CT imaging of carbonate rock with high porosity"*

### What's happening here

Regular search looks for exact words. Semantic search looks for *meaning*.
It works by converting text into a list of numbers called an **embedding vector**,
think of it as coordinates in a high-dimensional space where similar concepts
end up close together.

The process:
1. The query text is passed through an **embedding model** (in production, SambaNova)
   which outputs a vector like `[0.12, -0.45, 0.89, ...]` with hundreds of dimensions.
2. Neo4j stores a similar vector on each `DigitalDataset` node (pre-computed when the
   dataset was indexed).
3. `db.index.vector.queryNodes` finds the k nodes whose vectors are closest to the
   query vector using **cosine similarity** (a score from 0 to 1, higher = more similar).

This means a query about *"carbonate rock"* can match a dataset described as
*"limestone core sample"* even though those words are different, because the
embedding model knows they're related concepts.

**Note:** The placeholder vector below (`[0.1] * 768`) is just for testing.
In the real assistant, this is replaced with an actual embedding from SambaNova.

In [16]:
# NOTE: This query requires the "dataset-embeddings" vector index, which should exist
# on the knowledge graph but not on this local test database.
# I don't currently have VPN access to connect to the TACC VM, so this cell
# will raise an index-not-found error when run locally.
# Once VPN access is set up, this same code will run against the real graph
# and return actual similarity-ranked results, no code changes needed.

# The text we want to find similar datasets for
query_text = "micro-CT imaging of carbonate rock with high porosity"

# TODO: replace this placeholder with a real embedding call
#   query_embedding = sambanova_embed(query_text)
# The actual dimension must match what was used to build the Neo4j vector index (just using a common one for now).
query_embedding = [0.1] * 768

# k=5 means return the 5 most similar datasets.
# Returns a list of SearchResult objects (dataset_id, score, properties).
results = store.semantic_search(query_embedding, k=5)

print(f"Top {len(results)} results for: '{query_text}'\n")
for r in results:
    # score closer to 1.0 = more similar to the query
    print(f"  [{r.score:.3f}] {r.dataset_id} — {r.properties.get('rockType', 'N/A')}")

RuntimeError: Cypher execution failed: {code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `db.index.vector.queryNodes`: Caused by: java.lang.IllegalArgumentException: There is no such vector schema index: dataset-embeddings}

---
## Query 2 — Metadata Filter

**Intent:** Find datasets by *exact property values*, structured lookup.

> *"Show me all sandstone datasets that have been segmented"*

### What's happening here

This is the opposite of semantic search; instead of fuzzy similarity, we want
an exact match on specific fields. Think of it like a database WHERE clause.

`filter_by_metadata` takes a plain Python dict of `{field: value}` pairs and
dynamically builds a **parameterized Cypher query** like:
```
MATCH (n:DigitalDataset)
WHERE n.rockType = $param_rockType AND n.segmented = $param_segmented
RETURN n.id
```

The values are **parameterized** (passed separately from the query string) to prevent
Cypher injection attacks, the same reason you use `?` placeholders in SQL.
The keys are validated against a regex so nobody can sneak in a malicious field name.

The filter dict is intentionally open-ended, any field from the Croissant metadata
schema (like `license`, `distribution`, `recordSet`) works without changing the function.

In [17]:
# Any combination of DigitalDataset properties works here.
# Add or remove keys to change the filter, no code changes needed in the function itself.
filters = {
    "rockType":  "Sandstone",  # must match exactly (case-sensitive)
    "segmented": "true"        # stored as a string in the graph, not a boolean
}

# Returns a list of dataset ID strings (not full node objects).
# Use these IDs to look up full details if needed.
dataset_ids = store.filter_by_metadata(filters)

print(f"Datasets matching {filters}:\n")
for did in dataset_ids:
    print(f"  {did}")
print(f"\nTotal: {len(dataset_ids)}")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: id)} {position: line: 1, column: 105, offset: 104} for query: 'MATCH (n:DigitalDataset) WHERE n.rockType = $param_rockType AND n.segmented = $param_segmented RETURN n.id AS dataset_id'


Datasets matching {'rockType': 'Sandstone', 'segmented': 'true'}:

  None

Total: 1


---
## Query 3 — Domain Q&A

**Intent:** Answer a *conceptual question* about porous media science.

> *"What is the difference between voxel resolution and effective porosity?"*

### What's happening here

This intent doesn't touch the database at all, it's handled entirely by the LLM.
The assistant recognizes it as a knowledge question and routes it straight to
the language model, which answers from its training data.

In the full assistant this goes through a **domain Q&A chain**, basically a
system prompt that tells the LLM to answer as a porous media expert and stay
grounded in the DPM context. For the demo, the answer is hardcoded below
since the chain isn't wired up yet, swap it for the real call when it's ready.

In [13]:
import os
from openai import OpenAI

# This endpoint is OpenAI-compatible, so we use the same client library
# just pointed at the SambaNova/TACC base URL instead of OpenAI's.
client = OpenAI(
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL")
)

question = "What is the difference between voxel resolution and effective porosity in digital rock physics?"

response = client.chat.completions.create(
    model=os.getenv("LLM_MODEL"),
    messages=[
        {"role": "system", "content": "You are a domain expert in digital rock physics and porous media."},
        {"role": "user", "content": question}
    ]
)

answer = response.choices[0].message.content

print(f"Q: {question}\n")
print(f"A: {answer}")

Q: What is the difference between voxel resolution and effective porosity in digital rock physics?

A: In digital rock physics, voxel resolution and effective porosity are two distinct concepts that are crucial in understanding the properties of porous media, such as rocks.

**Voxel Resolution:**
Voxel resolution refers to the spatial resolution at which a digital rock is represented. A voxel (short for "volumetric pixel") is the smallest unit of a 3D digital image, representing a tiny cube of the rock's volume. The voxel resolution is typically measured in micrometers (μm) or pixels and determines the level of detail that can be captured in the digital representation of the rock. A higher voxel resolution means that more detailed information about the rock's microstructure can be obtained, such as the shape and size of pores, grains, and other features.

For example, a digital rock with a voxel resolution of 1 μm can capture features that are 1 μm in size, while a resolution of 10 μm 

---
## Query 4 — Workflow Guidance

**Intent:** Walk a user through *how to do something* in the DPM Portal.

> *"How do I publish a new dataset to the DPM Portal using Rocco?"*

### What's happening here

Similar to domain Q&A, no database query needed. The assistant recognizes
this as a *how-to* question and routes it to a workflow guidance chain.

The difference from domain Q&A is the *system prompt*, workflow guidance
is prompted to give step-by-step instructions grounded in how Rocco and the
DPM Portal actually work, rather than general scientific knowledge.

This is where Rocco's curation rubric is relevant, the assistant knows the
10 criteria Rocco evaluates and can guide users toward meeting all of them.

In [12]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL")
)

question = "How do I publish a new dataset to the DPM Portal using Rocco?"

response = client.chat.completions.create(
    model=os.getenv("LLM_MODEL"),
    messages=[
        {"role": "system", "content": "You are a helpful assistant that guides users through workflows for the Digital Porous Media (DPM) Portal and its AI curation tool, Rocco. Give clear, step-by-step instructions."},
        {"role": "user", "content": question}
    ]
)

answer = response.choices[0].message.content

print(f"Q: {question}\n")
print(f"A: {answer}")

Q: How do I publish a new dataset to the DPM Portal using Rocco?

A: Publishing a new dataset to the DPM Portal using Rocco involves several steps. Here's a step-by-step guide to help you through the process:

**Step 1: Prepare Your Dataset**

* Ensure your dataset is in a suitable format (e.g., CSV, Excel, or JSON) and is well-organized.
* Make sure your dataset includes all necessary metadata, such as:
	+ Dataset title and description
	+ Author information
	+ Keywords and categories
	+ Data source and methodology
* Validate your dataset for any errors or inconsistencies.

**Step 2: Log in to Rocco**

* Go to the Rocco website and log in with your credentials.
* If you don't have an account, create one by following the registration process.

**Step 3: Create a New Dataset Submission**

* Click on the "Submit Dataset" button on the Rocco dashboard.
* Fill in the required information, including:
	+ Dataset title and description
	+ Author information
	+ Keywords and categories
* Upload y

---
## Query 5 — Literature Lookup

**Intent:** Find datasets with linked publications and check PDF availability.

> *"Which datasets have associated published papers, and are the PDFs available?"*

### What's happening here

This one uses a **raw Cypher query** directly, no helper method needed because
it traverses a *relationship* between node types, which `filter_by_metadata`
doesn't handle (it only looks at properties on a single node type).

The query follows the graph pattern:
```
(DigitalDataset) -[:HAS_PUBLICATION]-> (RelatedPublication)
```
This is what makes a graph database powerful, it is easily traverseable and has connections
between entities naturally, in a way that would require a JOIN in SQL.

This query also directly feeds into the Week 2 audit, once I have
VPN access, this is the same pattern is how I'll count how many datasets have
linked papers and whether PDFs are available.

In [18]:
# Raw Cypher — MATCH finds the pattern, RETURN selects what we want back.
# The --> arrow means we follow the HAS_PUBLICATION relationship outward.
query = """
    MATCH (d:DigitalDataset)-[:HAS_PUBLICATION]->(p:RelatedPublication)
    RETURN d.id            AS dataset_id,
           p.title         AS paper_title,
           p.doi           AS doi,
           p.pdfAvailable  AS pdf_available
    ORDER BY pdf_available DESC
"""

# execute_cypher runs any raw Cypher string and returns a list of dicts
results = store.execute_cypher(query)

print(f"Datasets with linked publications: {len(results)}\n")
for r in results:
    pdf_status = "PDF available" if r.get("pdf_available") else "no PDF"
    print(f"  {r['dataset_id']} — {r['paper_title']} [{pdf_status}]")
    if r.get("doi"):
        print(f"    DOI: {r['doi']}")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: pdfAvailable)} {position: line: 6, column: 14, offset: 206} for query: '\n    MATCH (d:DigitalDataset)-[:HAS_PUBLICATION]->(p:RelatedPublication)\n    RETURN d.id            AS dataset_id,\n           p.title         AS paper_title,\n           p.doi           AS doi,\n           p.pdfAvailable  AS pdf_available\n    ORDER BY pdf_available DESC\n'
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database}

Datasets with linked publications: 0



---
## Bonus — Hybrid Search (Semantic + Metadata Combined)

**Intent:** Shows off `search_datasets`, which merges vector search and metadata
filtering into a *single* Cypher query.

> *"Find carbonate datasets similar to this query that have also been segmented"*

### What's happening here

This is the most powerful query type, it combines both intents at once.
Instead of running two separate queries and merging the results in Python,
`search_datasets` appends the WHERE clause directly to the vector index call
so Neo4j handles everything in one round-trip:

```cypher
CALL db.index.vector.queryNodes($index, $k, $embedding)
YIELD node AS n, score
WHERE n.segmented = $param_segmented
RETURN n.id, score, properties(n)
```

This is more efficient than two separate calls and means the score ranking
already accounts for the filter, you get the most similar datasets *among*
those that match the metadata criteria, not just all filtered datasets ranked afterward.

In [2]:
from sentence_transformers import SentenceTransformer

# NOTE: Using 'all-MiniLM-L6-v2' as a placeholder embedding model, it outputs
# 384-dimensional vectors. The real Neo4j vector index on the TACC graph may have
# been built with a different model (possibly 768-dim).
# TODO: confirm with mentor which embedding model was used to build the
# "dataset-embeddings" index so query vectors match its dimensionality.
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

query_text = "carbonate rock with high porosity"
query_embedding = embedding_model.encode(query_text).tolist()

# Combines vector similarity AND metadata filter in one DB call.
# Add more keys to filters to narrow results further.
results = store.search_datasets(
    query_embedding,
    filters={"segmented": "true"},
    k=5
)

print("Hybrid results (semantic similarity + segmented=true):\n")
for r in results:
    print(f"  [{r.score:.3f}] {r.dataset_id} — {r.properties.get('rockType', 'N/A')}")

NameError: name 'store' is not defined